# Dataset setup

This notebook (a) sets up access to **SportsMOT**, (b) generates a quick **synthetic** dataset to verify the training pipeline works end-to-end, and (c) gives a visual sanity check.

> **Important caveat.** SportsMOT does *not* annotate the ball — only players. Since our project is about ball-conditioned trajectory prediction, the project README lists alternative ball-aware sources (NBA SportVU 2015–16, SoccerNet-Tracking) and describes how to inject ball annotations from a separate detector. The synthetic dataset below has a ball by construction so we can validate models before real data is wired in.

## Imports & environment check

In [3]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from ballcond.utils import get_device
from ballcond.data.synthetic import synthesize_dataset

print('python', sys.version.split()[0])
print('torch ', torch.__version__, '   device:', get_device())

python 3.13.7
torch  2.11.0    device: mps


## SportsMOT download

SportsMOT ships through CodaLab. Easiest mirror is Hugging Face. The cell below uses `huggingface_hub` to fetch the train+val splits into `data/raw/SportsMOT/`. Skip this cell if you'd rather download the official zip from CodaLab manually.

It is large (~35 GB). For a first run, prefer the synthetic dataset section below.

In [ ]:
DOWNLOAD_SPORTSMOT = True  # flip to True once you're ready to spend ~35 GB

if DOWNLOAD_SPORTSMOT:
    from huggingface_hub import snapshot_download
    target = ROOT / 'data' / 'raw' / 'SportsMOT'
    target.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='MCG-NJU/SportsMOT',
        repo_type='dataset',
        local_dir=str(target),
        local_dir_use_symlinks=False,
    )
    print('Downloaded SportsMOT to', target)
else:
    print('Skipping SportsMOT download (set DOWNLOAD_SPORTSMOT=True to fetch).')

/Users/atharvjain/code/cv/project/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Once SportsMOT is on disk, you can load a single clip with `load_sportsmot_sequence` to verify the loader. Note that `Sequence.ball is None` (the dataset only labels players).

In [ ]:
from ballcond.data.sportsmot import load_sportsmot_split

split_root = ROOT / 'data' / 'raw' / 'SportsMOT' / 'dataset' / 'train'
if split_root.exists():
    seqs = load_sportsmot_split(split_root, limit=2)
    for s in seqs:
        print(s.sequence_id, s.sport, s.num_frames, 'frames,', s.num_players, 'players, has_ball=', s.has_ball)
else:
    print('SportsMOT not yet downloaded at', split_root)

## Synthetic dataset preview

The synthetic generator produces clips where ten players each have a home cell and are pulled toward a smoothly drifting ball. This mirrors the regime the paper argues for — information flowing from the ball to the players — and lets us evaluate whether ball conditioning helps under known ground truth.

Run the cell below to generate a small dataset and visualize one clip.

In [ ]:
seqs = synthesize_dataset(n=4, T=200, seed=0)
print(f'Generated {len(seqs)} synthetic clips')
for s in seqs[:2]:
    print(' ', s.sequence_id, '   players:', s.num_players, '   ball:', s.has_ball)

In [ ]:
seq = seqs[0]
fig, ax = plt.subplots(figsize=(6, 6))
for n in range(seq.num_players):
    ax.plot(seq.players[:, n, 0], seq.players[:, n, 1], alpha=0.6, label=f'p{n}')
ax.plot(seq.ball[:, 0], seq.ball[:, 1], color='black', linewidth=2.5, label='ball')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.set_title(f'Synthetic clip: {seq.sequence_id}')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

## Sanity-check the windowed dataset

Build a `WindowDataset` and inspect one batch. This is what every model in the project consumes.

In [ ]:
from torch.utils.data import DataLoader
from ballcond.data import WindowDataset, collate_windows

ds = WindowDataset(seqs, history=20, horizon=10, stride=5)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate_windows)
batch = next(iter(loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'{k:24s} {tuple(v.shape)} {v.dtype}')
    else:
        print(f'{k:24s} {type(v).__name__}')

## Next steps

Run the synthetic sweep from the project root:

```bash
bash scripts/run_synthetic_sweep.sh
```

This trains the Kalman, LSTM, symmetric-transformer, and ball-conditioned-transformer models and dumps ADE/FDE per horizon to `results/<run_name>/metrics.json`.